# Working with parquet files

## Objective

+ In this assignment, we will use the data downloaded with the module `data_manager` to create features.

(11 pts total)

## Prerequisites

+ This notebook assumes that price data is available to you in the environment variable `PRICE_DATA`. If you have not done so, then execute the notebook `01_materials/labs/2_data_engineering.ipynb` to create this data set.


+ Load the environment variables using dotenv. (1 pt)

In [1]:
# Write your code below.
%load_ext dotenv 
%dotenv 

In [3]:
import dask.dataframe as dd

+ Load the environment variable `PRICE_DATA`.
+ Use [glob](https://docs.python.org/3/library/glob.html) to find the path of all parquet files in the directory `PRICE_DATA`.

(1pt)

In [14]:
import os
from glob import glob

#load price data
PRICE_DATA = os.getenv('PRICE_DATA')
#sanity check
print(PRICE_DATA)

# find the path of all files
parquet_files = glob(os.path.join(PRICE_DATA,"**/*.parquet"), recursive = True)
#sanity check 
len(parquet_files)



../../05_src/data/prices/


4698

For each ticker and using Dask, do the following:

+ Add lags for variables Close and Adj_Close.
+ Add returns based on Close:
    
    - `returns`: (Close / Close_lag_1) - 1

+ Add the following range: 

    - `hi_lo_range`: this is the day's High minus Low.

+ Assign the result to `dd_feat`.

(4 pt)

In [15]:
# read parquet files  and organize them by ticker 
dd_px = dd.read_parquet(parquet_files).set_index("ticker")

In [17]:
#sanity check again ( lol always)
dd_px.head()

,Date,Open,High,Low,Close,Adj Close,Volume,source,Year
ticker,,,,,,,,,
A,1999-11-18,32.546494,35.765381,28.612303,31.473534,27.068665,62546300.0,A.csv,1999
A,1999-11-19,30.713520,30.758226,28.478184,28.880543,24.838577,15234100.0,A.csv,1999
A,1999-11-22,29.551144,31.473534,28.657009,31.473534,27.068665,6577800.0,A.csv,1999
A,1999-11-23,30.400572,31.205294,28.612303,28.612303,24.607880,5975600.0,A.csv,1999
A,1999-11-24,28.701717,29.998211,28.612303,29.372318,25.261524,4843200.0,A.csv,1999


In [ ]:
# need to import pandas... was running into errors
import pandas as pd 

In [ ]:
#group the data by ticker
dd_feat = (
    dd_px
    .groupby('ticker',group_keys =False)
    .apply(
        lambda x: x.sort_values('Date', ascending = True)
        #assign a new variable Close_lag_1 by shifting Close by one position
         .assign(
             Close_lag_1 = x['Close'].shift(1), 
             #same thing for Adj_Close_lag_1 by shifting Adj_Close by one position
             Adj_Close_lag_1 = x['Adj Close'].shift(1),
              #add return variable
              returns = lambda x: x['Close'] / x['Close_lag_1'] - 1,
              #add range
              hi_lo_range = lambda x: x['High'] - x['Low']
         ),
         meta = pd.DataFrame(data ={
             'Date': 'datetime64[ns]',
             'Open': 'f8',
             'High': 'f8',
             'Low': 'f8',
             'Close': 'f8',
             'Adj Close': 'f8',
             'Volume': 'i8',
             'source': 'object',
             'Year': 'int32',
             'Close_lag_1': 'f8',
             'Adj_Close_lag_1': 'f8',
             'returns':'f8',
             'hi_lo_range' : 'f8'},
             index = pd.Index([], dtype=pd.StringDtype(), name='ticker'))
    )
)   

#sanity checks
dd_feat.head()

,Date,Open,High,Low,Close,Adj Close,Volume,source,Year,Close_lag_1,Adj_Close_lag_1,returns,hi_lo_range
ticker,,,,,,,,,,,,,
A,1999-11-18,32.546494,35.765381,28.612303,31.473534,27.068665,62546300.0,A.csv,1999,NaN,NaN,NaN,7.153078
A,1999-11-19,30.713520,30.758226,28.478184,28.880543,24.838577,15234100.0,A.csv,1999,31.473534,27.068665,-0.082386,2.280043
A,1999-11-22,29.551144,31.473534,28.657009,31.473534,27.068665,6577800.0,A.csv,1999,28.880543,24.838577,0.089783,2.816525
A,1999-11-23,30.400572,31.205294,28.612303,28.612303,24.607880,5975600.0,A.csv,1999,31.473534,27.068665,-0.090909,2.592991
A,1999-11-24,28.701717,29.998211,28.612303,29.372318,25.261524,4843200.0,A.csv,1999,28.612303,24.607880,0.026563,1.385908


+ Convert the Dask data frame to a pandas data frame. 
+ Add a new feature containing the moving average of `returns` using a window of 10 days. There are several ways to solve this task, a simple one uses `.rolling(10).mean()`.

(3 pt)

In [ ]:
# I think I already converted the dask dataframe into pandas on the code above. 
#moving on to adding moving average of returns 
dd_feat_mv = dd_feat.assign(
    mv_returns = lambda x: x['returns'].rolling(10).mean()
    )


,Date,Open,High,Low,Close,Adj Close,Volume,source,Year,Close_lag_1,Adj_Close_lag_1,returns,hi_lo_range,mv_returns
ticker,,,,,,,,,,,,,,
A,1999-11-18,32.546494,35.765381,28.612303,31.473534,27.068665,62546300.0,A.csv,1999,NaN,NaN,NaN,7.153078,NaN
A,1999-11-19,30.713520,30.758226,28.478184,28.880543,24.838577,15234100.0,A.csv,1999,31.473534,27.068665,-0.082386,2.280043,NaN
A,1999-11-22,29.551144,31.473534,28.657009,31.473534,27.068665,6577800.0,A.csv,1999,28.880543,24.838577,0.089783,2.816525,NaN
A,1999-11-23,30.400572,31.205294,28.612303,28.612303,24.607880,5975600.0,A.csv,1999,31.473534,27.068665,-0.090909,2.592991,NaN
A,1999-11-24,28.701717,29.998211,28.612303,29.372318,25.261524,4843200.0,A.csv,1999,28.612303,24.607880,0.026563,1.385908,NaN


Please comment:

+ Was it necessary to convert to pandas to calculate the moving average return?
Hm... Although Dask is based on lazy execution, I don't believe that it prevents to calculate .rolling().mean. Perhaps it will then only execute when we compute it? 

+ Would it have been better to do it in Dask? Why?
Yes, for memory efficiency and parallel processing

(1 pt)

## Criteria

The [rubric](./assignment_1_rubric_clean.xlsx) contains the criteria for grading.

## Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

### Submission Parameters:
* Submission Due Date: `HH:MM AM/PM - DD/MM/YYYY`
* The branch name for your repo should be: `assignment-1`
* What to submit for this assignment:
    * This Jupyter Notebook (assignment_1.ipynb) should be populated and should be the only change in your pull request.
* What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    * Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

Checklist:
- [ x] Created a branch with the correct naming convention.
- [x ] Ensured that the repository is public.
- [x ] Reviewed the PR description guidelines and adhered to them.
- [ x] Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack at `#cohort-3-help`. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.